In [36]:
import re
from datetime import datetime, timedelta
from typing import Iterable, List, Dict, Optional

# Target markers (cheap substring checks first)
START_SUBSTR = "Starting evaluation for instance"
BEGIN_SUBSTR = "BEGIN Runtime Initialization Fn"
END_SUBSTR   = "END Runtime Completion Fn"

# Keep existing robust patterns
START_RE = re.compile(r'Starting evaluation for instance\s+(?P<id>.+?)\.')
BEGIN_RE = re.compile(r'\bBEGIN Runtime Initialization Fn\b')
END_RE   = re.compile(r'\bEND Runtime Completion Fn\b')

# Fast timestamp-at-line-start with optional ANSI color codes
# e.g. "\x1b[92m02:18:18" or "02:18:18"
TS_AT_START_RE = re.compile(
    r'^\s*(?:\x1b\[[0-9;]*m\s*)*(?P<h>\d{2}):(?P<m>\d{2}):(?P<s>\d{2})'
)

def _parse_time(h: int, m: int, s: int) -> datetime:
    """Force same date for all times."""
    return datetime(1970, 1, 1, h, m, s)

def _extract_timestamp_fast(line: str) -> Optional[datetime]:
    """
    Only parse HH:MM:SS at the *start* of the line (after optional ANSI codes).
    Returns datetime(1970-01-01 HH:MM:SS) or None.
    """
    m = TS_AT_START_RE.match(line)
    if not m:
        return None
    return _parse_time(int(m.group('h')), int(m.group('m')), int(m.group('s')))

def parse_log_triplets(lines: Iterable[str], to_dataframe: bool = False):
    """
    Only parse timestamps for lines that contain one of the target markers.
    Perf-friendly: cheap substring checks first, then minimal regex work.
    """
    results: List[Dict] = []
    current = None  # {'id','t1','t2','t3'}

    for line in lines:
        # Cheap pre-filter: skip most lines fast
        if (START_SUBSTR not in line
            and BEGIN_SUBSTR not in line
            and END_SUBSTR   not in line):
            continue

        # Try to extract timestamp only now
        ts = _extract_timestamp_fast(line)
        if ts is None:
            # If a target line lacks a parsable timestamp, skip (or log/warn)
            continue

        # 1) Starting evaluation...
        if START_SUBSTR in line:
            m_start = START_RE.search(line)
            if m_start:
                # Flush partial previous instance if needed
                if current and ('t1' in current) and ('t2' in current) and ('t3' not in current):
                    results.append({
                        'instance_id': current['id'],
                        'docker_build_start_ts': current['t1'],
                        'llm_execution_start_ts': current.get('t2'),
                        'end_ts': None,
                        'docker_build_time': (current['t2'] - current['t1']).total_seconds() if current.get('t2') else None,
                        'llm_execution_time': None,
                        'total_time': None,
                        'complete_triplet': False,
                    })
                current = {'id': m_start.group('id'), 't1': ts}
                continue

        # 2) BEGIN Runtime Initialization Fn
        if BEGIN_SUBSTR in line and BEGIN_RE.search(line):
            if current and 't1' in current and 't2' not in current:
                current['t2'] = ts
            continue

        # 3) END Runtime Completion Fn
        if END_SUBSTR in line and END_RE.search(line):
            if current and 't1' in current and 't3' not in current:
                current['t3'] = ts
                results.append({
                    'instance_id': current['id'],
                    'docker_build_start_ts': current['t1'],
                    'llm_execution_start_ts': current.get('t2'),
                    'end_ts': current.get('t3'),
                    'docker_build_time': (current['t2'] - current['t1']).total_seconds() if current.get('t2') else None,
                    'llm_execution_time': (current['t3'] - current['t2']).total_seconds() if current.get('t2') and current.get('t3') else None,
                    'total_time': (current['t3'] - current['t1']).total_seconds() if current.get('t1') and current.get('t3') else None,
                    'complete_triplet': ('t2' in current),
                })
                current = None
            continue

    # EOF: flush partial
    if current:
        results.append({
            'instance_id': current['id'],
            'docker_build_start_ts': current.get('t1'),
            'llm_execution_start_ts': current.get('t2'),
            'end_ts': current.get('t3'),
            'docker_build_time': (current['t2'] - current['t1']).total_seconds() if current.get('t2') and current.get('t1') else None,
            'llm_execution_time': (current['t3'] - current['t2']).total_seconds() if current.get('t3') and current.get('t2') else None,
            'total_time': (current['t3'] - current['t1']).total_seconds() if current.get('t3') and current.get('t1') else None,
            'complete_triplet': ('t1' in current) and ('t2' in current) and ('t3' in current),
        })

    if to_dataframe:
        try:
            import pandas as pd
            df = pd.DataFrame(results)
            cols = [
                'instance_id', 'docker_build_start_ts', 'llm_execution_start_ts', 'end_ts',
                'docker_build_time', 'llm_execution_time', 'total_time', 'complete_triplet'
            ]
            return df[cols]
        except ImportError:
            pass
    return results

def parse_log_file(path: str, to_dataframe: bool = False):
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        return parse_log_triplets(f, to_dataframe=to_dataframe)


In [37]:
docker_local_log_file = '/workspaces/OpenHands/qwen-coder-30b-small_maxiter_100_N_local_docker_test_10_samples.log'
docker_remote_log_file = '/workspaces/OpenHands/qwen-coder-30b-small_maxiter_100_N_remote_docker_test_10_samples.log'

In [38]:
local_df = parse_log_file(docker_local_log_file, to_dataframe=True)
local_df.head(20)

,instance_id,docker_build_start_ts,llm_execution_start_ts,end_ts,docker_build_time,llm_execution_time,total_time,complete_triplet
0,pyvista__pyvista-432,1970-01-01 02:13:42,1970-01-01 02:14:29,1970-01-01 02:17:50,47.0,201.0,248.0,True
1,sqlfluff__sqlfluff-891,1970-01-01 02:18:00,1970-01-01 02:22:40,1970-01-01 02:26:37,280.0,237.0,517.0,True
2,pvlib__pvlib-python-1666,1970-01-01 02:27:00,1970-01-01 02:31:11,1970-01-01 02:34:12,251.0,181.0,432.0,True
3,sqlfluff__sqlfluff-880,1970-01-01 02:34:21,1970-01-01 02:38:35,1970-01-01 02:41:38,254.0,183.0,437.0,True
4,pylint-dev__astroid-2015,1970-01-01 02:41:49,1970-01-01 02:46:20,1970-01-01 03:07:55,271.0,1295.0,1566.0,True
5,marshmallow-code__marshmallow-2123,1970-01-01 03:08:08,1970-01-01 03:12:31,1970-01-01 03:15:46,263.0,195.0,458.0,True
6,pyvista__pyvista-3675,1970-01-01 03:16:44,1970-01-01 03:22:16,1970-01-01 03:25:30,332.0,194.0,526.0,True
7,pvlib__pvlib-python-1176,1970-01-01 03:25:55,1970-01-01 03:30:35,1970-01-01 03:34:41,280.0,246.0,526.0,True
8,pvlib__pvlib-python-1623,1970-01-01 03:35:02,1970-01-01 03:40:04,1970-01-01 03:42:32,302.0,148.0,450.0,True
9,pvlib__pvlib-python-1478,1970-01-01 03:42:55,1970-01-01 03:47:28,1970-01-01 03:52:17,273.0,289.0,562.0,True


In [39]:
remote_df = parse_log_file(docker_remote_log_file, to_dataframe=True)
remote_df.head(20)

,instance_id,docker_build_start_ts,llm_execution_start_ts,end_ts,docker_build_time,llm_execution_time,total_time,complete_triplet
0,pyvista__pyvista-432,1970-01-01 03:11:31,1970-01-01 03:15:51,1970-01-01 03:19:11,260.0,200.0,460.0,True
1,sqlfluff__sqlfluff-891,1970-01-01 03:19:15,1970-01-01 03:23:46,1970-01-01 03:28:44,271.0,298.0,569.0,True
2,pvlib__pvlib-python-1666,1970-01-01 03:28:51,1970-01-01 03:33:16,1970-01-01 03:36:19,265.0,183.0,448.0,True
3,sqlfluff__sqlfluff-880,1970-01-01 03:36:21,1970-01-01 03:40:25,1970-01-01 03:43:46,244.0,201.0,445.0,True
4,pylint-dev__astroid-2015,1970-01-01 03:43:52,1970-01-01 03:48:14,1970-01-01 03:57:06,262.0,532.0,794.0,True
5,marshmallow-code__marshmallow-2123,1970-01-01 04:00:46,1970-01-01 04:01:18,1970-01-01 04:04:16,32.0,178.0,210.0,True
6,pyvista__pyvista-3675,1970-01-01 04:04:20,1970-01-01 04:07:38,1970-01-01 04:10:58,198.0,200.0,398.0,True
7,pvlib__pvlib-python-1176,1970-01-01 04:11:01,1970-01-01 04:14:06,1970-01-01 04:18:02,185.0,236.0,421.0,True
8,pvlib__pvlib-python-1623,1970-01-01 04:18:03,1970-01-01 04:21:02,1970-01-01 04:23:38,179.0,156.0,335.0,True
9,pvlib__pvlib-python-1478,1970-01-01 04:23:41,1970-01-01 04:28:03,1970-01-01 04:32:43,262.0,280.0,542.0,True


In [43]:
# Total times across all instances
local_totals = local_df[["docker_build_time", "llm_execution_time", "total_time"]].sum()
remote_totals = remote_df[["docker_build_time", "llm_execution_time", "total_time"]].sum()

print("Local totals (minutes):")
print(local_totals / 60)
print("\nRemote totals (minutes):")
print(remote_totals / 60)


Local totals (minutes):
docker_build_time     42.550000
llm_execution_time    52.816667
total_time            95.366667
dtype: float64

Remote totals (minutes):
docker_build_time     35.966667
llm_execution_time    41.066667
total_time            77.033333
dtype: float64
